# ATLAS — Teste rápido das 12 PROMISSORAS em H100 (~15 min)

Este notebook testa **todos os mecanismos promissores** descobertos na pesquisa ATLAS (FashionMNIST) contra o **baseline na mesma escala**, em uma GPU H100 do Google Colab.

**Como usar:**
1. Runtime → Change runtime type → **GPU (H100)** (A100 também funciona)
2. Runtime → Run all
3. Aguarde ~15 min; o ranking final aparece na última célula

**O que é testado (10 configs × 3 seeds = 30 treinos):**

| Config | Origem | Tipo |
|--------|--------|------|
| baseline | — | régua de comparação |
| warmup+cosine | exp-007/011 | RECOMB |
| local_blend | exp-016/018 | NOVEL |
| local_blend_k5 | exp-021/026 | NOVEL |
| laplacian_blend | exp-022/028 | NOVEL |
| blend+warmup+cosine | exp-024 | RECOMB |
| blend+label_smoothing | exp-025/027 | RECOMB (ex-campeão) |
| k5+label_smoothing | exp-032 | RECOMB |
| multi_scale+LS+wc | exp-042/047 | NOVEL (campeão) |
| tri_scale+LS+wc | exp-053 | NOVEL (recorde 92.98% CPU) |

**Escala vs testes originais em CPU:** modelo 2× mais largo (64/128 canais), batch 512 (4×), dados residentes na GPU (zero gargalo de I/O), bf16 autocast.

**Regra de honestidade ATLAS:** todos os números vêm de execução real nesta sessão; comparação sempre vs baseline no MESMO budget; 3 seeds por config.

In [ ]:
# @title 1. Verificação da GPU
import torch

assert torch.cuda.is_available(), "GPU indisponível! Runtime → Change runtime type → GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu}")
print(f"VRAM: {vram:.0f} GB")
print(f"PyTorch: {torch.__version__}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda")

In [ ]:
# @title 2. Dataset FashionMNIST — carregado inteiro na GPU (zero gargalo de I/O)
from torchvision import datasets, transforms

tfm = transforms.ToTensor()
train_ds = datasets.FashionMNIST("./data", train=True, download=True, transform=tfm)
val_ds = datasets.FashionMNIST("./data", train=False, download=True, transform=tfm)

MEAN, STD = 0.2860, 0.3530

def to_gpu_tensors(ds):
    x = ds.data.unsqueeze(1).float().div_(255.0).sub_(MEAN).div_(STD)
    y = ds.targets.clone()
    return x.to(DEVICE), y.to(DEVICE)

X_TRAIN, Y_TRAIN = to_gpu_tensors(train_ds)
X_VAL, Y_VAL = to_gpu_tensors(val_ds)
print(f"train: {tuple(X_TRAIN.shape)} | val: {tuple(X_VAL.shape)} | tudo na GPU")

In [ ]:
# @title 3. Mecanismos ATLAS (código idêntico ao da pesquisa CPU, portado para GPU)
import math
import random
import time

import torch.nn as nn
import torch.nn.functional as F


class LocalBlend(nn.Module):
    """NOVEL ATLAS: mistura local depthwise com gate escalar por canal.

    gate = sigmoid(média espacial por canal)
    out  = gate * depthwise_conv(x) + (1-gate) * x
    """

    def __init__(self, channels: int, kernel: int = 3, gate_mode: str = "mean") -> None:
        super().__init__()
        pad = kernel // 2
        self.dw = nn.Conv2d(channels, channels, kernel, padding=pad, groups=channels, bias=False)
        if gate_mode == "laplacian":
            w = torch.tensor([[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]])
            self.dw.weight.data = w.view(1, 1, 3, 3).repeat(channels, 1, 1, 1)
        else:
            nn.init.dirac_(self.dw.weight)
        self.gate_mode = gate_mode

    def forward(self, x):
        local = self.dw(x)
        gate = torch.sigmoid(x.mean(dim=(2, 3), keepdim=True))
        return gate * local + (1.0 - gate) * x


class MultiScaleBlend(nn.Module):
    """NOVEL ATLAS: fusão depthwise 3x3 + 5x5 com gate de média espacial."""

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.dw3 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.dw5 = nn.Conv2d(channels, channels, 5, padding=2, groups=channels, bias=False)
        nn.init.dirac_(self.dw3.weight)
        nn.init.dirac_(self.dw5.weight)

    def forward(self, x):
        local = 0.5 * self.dw3(x) + 0.5 * self.dw5(x)
        gate = torch.sigmoid(x.mean(dim=(2, 3), keepdim=True))
        return gate * local + (1.0 - gate) * x


class TriScaleBlend(nn.Module):
    """NOVEL ATLAS (recordista CPU 92.98%): fusão depthwise 3+5+7 com gate."""

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.dw3 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.dw5 = nn.Conv2d(channels, channels, 5, padding=2, groups=channels, bias=False)
        self.dw7 = nn.Conv2d(channels, channels, 7, padding=3, groups=channels, bias=False)
        for m in (self.dw3, self.dw5, self.dw7):
            nn.init.dirac_(m.weight)

    def forward(self, x):
        local = (self.dw3(x) + self.dw5(x) + self.dw7(x)) / 3.0
        gate = torch.sigmoid(x.mean(dim=(2, 3), keepdim=True))
        return gate * local + (1.0 - gate) * x


def make_mixing(name: str, channels: int) -> nn.Module:
    if name == "local_blend":
        return LocalBlend(channels, kernel=3)
    if name == "local_blend_k5":
        return LocalBlend(channels, kernel=5)
    if name == "laplacian_blend":
        return LocalBlend(channels, kernel=3, gate_mode="laplacian")
    if name == "multi_scale_blend":
        return MultiScaleBlend(channels)
    if name == "tri_scale_blend":
        return TriScaleBlend(channels)
    return nn.Identity()


class SmallCNN(nn.Module):
    """Mesma arquitetura da pesquisa ATLAS, width_mult=2 (64/128 canais)."""

    def __init__(self, mixing: str = "none", width_mult: float = 2.0, hidden_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        c1, c2 = int(32 * width_mult), int(64 * width_mult)
        self.conv1 = nn.Conv2d(1, c1, 3, padding=1)
        self.conv2 = nn.Conv2d(c1, c2, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.mix1 = make_mixing(mixing, c1)
        self.mix2 = make_mixing(mixing, c2)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(c2 * 7 * 7, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 10)
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        # o kaiming acima sobrescreveu o init especial (dirac/laplacian) dos blends;
        # recria os módulos de mixing para restaurar o init correto
        self.mix1 = make_mixing(mixing, c1)
        self.mix2 = make_mixing(mixing, c2)

    def forward(self, x):
        x = self.pool(self.act(self.mix1(self.conv1(x))))
        x = self.pool(self.act(self.mix2(self.conv2(x))))
        x = x.flatten(1)
        x = self.dropout(self.act(self.fc1(x)))
        return self.fc2(x)


def lr_at_step(base_lr, step, total, warmup, scheduler):
    if warmup > 0 and step < warmup:
        return base_lr * (step + 1) / warmup
    if scheduler == "cosine":
        progress = (step - warmup) / max(1, total - warmup)
        return base_lr * 0.5 * (1 + math.cos(math.pi * progress))
    return base_lr


print("Mecanismos ATLAS carregados.")

In [ ]:
# @title 4. Loop de treino GPU (dados residentes, bf16 autocast)

BATCH = 512
N_TRAIN = X_TRAIN.size(0)


@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0
    for i in range(0, X_VAL.size(0), 2048):
        x, y = X_VAL[i : i + 2048], Y_VAL[i : i + 2048]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(x)
        correct += (logits.argmax(1) == y).sum().item()
    return correct / X_VAL.size(0)


def train_one(mixing="none", label_smoothing=0.0, warmup=0, scheduler="none",
              steps=3000, lr=2e-3, seed=0, log=False):
    torch.manual_seed(seed)
    random.seed(seed)
    model = SmallCNN(mixing=mixing).to(DEVICE).to(memory_format=torch.channels_last)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_acc = 0.0
    eval_every = max(1, steps // 4)
    t0 = time.perf_counter()
    model.train()
    for step in range(steps):
        idx = torch.randint(0, N_TRAIN, (BATCH,), device=DEVICE)
        x = X_TRAIN[idx].to(memory_format=torch.channels_last)
        y = Y_TRAIN[idx]

        for pg in opt.param_groups:
            pg["lr"] = lr_at_step(lr, step, steps, warmup, scheduler)

        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = F.cross_entropy(model(x), y, label_smoothing=label_smoothing)
        loss.backward()
        opt.step()

        if (step + 1) % eval_every == 0 or step == steps - 1:
            acc = evaluate(model)
            best_acc = max(best_acc, acc)
            model.train()
            if log:
                print(f"  step {step+1}/{steps}: val {acc*100:.2f}%")

    wall = time.perf_counter() - t0
    return best_acc, wall


# Calibração: mede a velocidade para ajustar steps ao orçamento de 15 min
print("Calibrando velocidade da GPU (200 steps)...")
_, t_cal = train_one(mixing="tri_scale_blend", steps=200, seed=0)
sps = 200 / t_cal
print(f"~{sps:.0f} steps/s no config mais pesado (tri_scale)")

# Orçamento: 30 treinos em ~13 min úteis => ~26s/treino no config mais pesado
STEPS = int(min(4000, max(1500, sps * 26)))
print(f"Steps por treino: {STEPS} (batch {BATCH} => {STEPS*BATCH/60000:.0f} épocas)")

In [ ]:
# @title 5. Executa as 10 configs × 3 seeds (~13 min)
from statistics import mean, stdev

CONFIGS = {
    "baseline":            dict(),
    "warmup_cosine":       dict(warmup=200, scheduler="cosine"),
    "local_blend":         dict(mixing="local_blend"),
    "local_blend_k5":      dict(mixing="local_blend_k5"),
    "laplacian_blend":     dict(mixing="laplacian_blend"),
    "blend_wc":            dict(mixing="local_blend", warmup=200, scheduler="cosine"),
    "blend_ls":            dict(mixing="local_blend", label_smoothing=0.1),
    "k5_ls":               dict(mixing="local_blend_k5", label_smoothing=0.1),
    "multi_scale_ls_wc":   dict(mixing="multi_scale_blend", label_smoothing=0.1, warmup=200, scheduler="cosine"),
    "tri_scale_ls_wc":     dict(mixing="tri_scale_blend", label_smoothing=0.1, warmup=200, scheduler="cosine"),
}

SEEDS = [1000, 1001, 1002]
results = {}
t_batch = time.perf_counter()

for name, kw in CONFIGS.items():
    accs, walls = [], []
    for seed in SEEDS:
        acc, wall = train_one(steps=STEPS, seed=seed, **kw)
        accs.append(acc)
        walls.append(wall)
    results[name] = (mean(accs), stdev(accs), accs, mean(walls))
    elapsed = time.perf_counter() - t_batch
    print(f"[{elapsed/60:5.1f} min] {name:20} {mean(accs)*100:.2f}% ± {stdev(accs)*100:.2f}%  "
          f"(seeds: {[f'{a*100:.2f}' for a in accs]})")

print(f"\nTempo total: {(time.perf_counter()-t_batch)/60:.1f} min")

In [ ]:
# @title 6. Ranking final + veredito vs baseline
base_mean, base_std, _, _ = results["baseline"]
noise = 2 * base_std  # limiar de significância: 2x o desvio do baseline

print("=" * 78)
print(f"{'RANKING':22} {'ACC':>8} {'±STD':>7} {'Δ BASELINE':>11}  VEREDITO")
print("=" * 78)
for name, (m, s, accs, w) in sorted(results.items(), key=lambda kv: -kv[1][0]):
    delta = (m - base_mean) * 100
    if name == "baseline":
        verdict = "(régua)"
    elif delta > noise * 100:
        verdict = "CONFIRMADA — ganho significante"
    elif delta > 0:
        verdict = "marginal (dentro do ruído)"
    else:
        verdict = "REFUTADA nesta escala"
    print(f"{name:22} {m*100:7.2f}% {s*100:6.2f}% {delta:+10.2f}pp  {verdict}")

print("=" * 78)
print(f"\nBaseline: {base_mean*100:.2f}% ± {base_std*100:.2f}% | limiar significância: {noise*100:.2f}pp")
print(f"GPU: {gpu} | steps/treino: {STEPS} | batch: {BATCH} | 3 seeds")
print("\nInterpretação:")
print("- CONFIRMADA: o mecanismo mantém vantagem real sobre baseline em escala GPU")
print("- marginal: diferença menor que 2x o ruído entre seeds — não conclusivo")
print("- REFUTADA: em escala maior, o baseline alcança/supera o mecanismo")